In [ ]:
# !pip install beautifulsoup4

  Using cached beautifulsoup4-4.12.3-py3-none-any.whl.metadata (3.8 kB)
Using cached beautifulsoup4-4.12.3-py3-none-any.whl (147 kB)


In [13]:
import os
import json
import time
import random
import zipfile
import requests
import pandas as pd
from bs4 import BeautifulSoup

# Class Explanation: `NewsScraper`

## Overview

The `NewsScraper` class is designed for scraping news articles from three different Urdu news websites: Geo, Jang, and Express. The class has methods that cater to each site's unique structure and requirements. Below, we will go through the class and its methods, detailing what each function does, the input it takes, and the output it returns.

## Class Definition

```python
class NewsScraper:
    def __init__(self, id_=0):
        self.id = id_
```

## Method 1: `get_express_articles`

### Description

Scrapes news articles from the Express website across categories like saqafat (entertainment), business, sports, science-technology, and world. The method navigates through multiple pages for each category to gather a more extensive dataset.

### Input

- **`max_pages`**: The number of pages to scrape for each category (default is 7).

### Process

- Iterates over each category and page.
- Requests each category page and finds article cards within `<ul class='tedit-shortnews listing-page'>`.
- Extracts the article's headline, link, and content by navigating through `<div class='horiz-news3-caption'>` and `<span class='story-text'>`.

### Output

- **Returns**: A tuple of:
  - A Pandas DataFrame containing columns: `id`, `title`, and `link`).
  - A dictionary `express_contents` where the key is the article ID and the value is the article content.

### Data Structure

- Article cards are identified by `<li>` tags.
- Content is structured within `<span class='story-text'>` and `<p>` tags.


In [14]:
class NewsScraper:
    def __init__(self):
        self.id = 0

    def get_express_articles(self, max_pages=14):
        express_df = {
            "id": [],
            "title": [],
            "link": [],
            "content": [],
            "gold_label": [],
        }
        base_url = 'https://www.express.pk'
        categories = ['saqafat', 'business', 'sports', 'science', 'world']
        for category in categories:
            for page in range(1, max_pages + 1):
                print(f"Scraping page {page} of category '{category}'...")
                url = f"{base_url}/{category}/archives?page={page}"
                response = requests.get(url)
                if response.url != url:
                    print(f"Redirected from {url} to {response.url}")
                response.raise_for_status()
                soup = BeautifulSoup(response.text, "html.parser")
                cards = soup.find(
                    'ul', class_='tedit-shortnews listing-page').find_all('li')
                print(
                    f"\t--> Found {len(cards)} articles on page {page} of '{category}'.")
                success_count = 0
                for card in cards:
                    try:
                        div = card.find('div', class_='horiz-news3-caption')
                        headline = div.find('a').get_text(
                            strip=True).replace('\xa0', ' ')
                        link = div.find('a')['href']
                        article_response = requests.get(link)
                        article_response.raise_for_status()
                        content_soup = BeautifulSoup(
                            article_response.text, "html.parser")
                        paras = content_soup.find(
                            'span', class_='story-text').find_all('p')
                        combined_text = " ".join(
                            p.get_text(strip=True).replace(
                                '\xa0', ' ').replace('\u200b', '')
                            for p in paras if p.get_text(strip=True)
                        )
                        express_df['id'].append(self.id)
                        express_df['title'].append(headline)
                        express_df['link'].append(link)
                        express_df['gold_label'].append(category.replace(
                            'saqafat', 'entertainment').replace('science', 'science-technology'))
                        express_df['content'].append(combined_text)
                        self.id += 1
                        success_count += 1
                    except Exception as e:
                        print(
                            f"\t--> Failed to scrape an article on page {page} of '{category}': {e}")
                print(
                    f"\t--> Successfully scraped {success_count} articles from page {page} of '{category}'.")
            print('')
        return pd.DataFrame(express_df)

    def get_dunya_articles(self, max_pages=1):
        dunya_df = {
            "id": [],
            "title": [],
            "link": [],
            "content": [],
            "gold_label": [],
        }
        base_url = 'https://urdu.dunyanews.tv/'
        categories = ['Entertainment', 'Business',
                      'Sports', 'Technology', 'World']
        for category in categories:
            for page in range(1, max_pages + 1):
                print(f"Scraping page {page} of category '{category}'...")
                url = f"{base_url}/index.php/ur/{category}"
                response = requests.get(url)
                if response.url != url:
                    print(f"Redirected from {url} to {response.url}")
                response.raise_for_status()
                soup = BeautifulSoup(response.text, "html.parser")
                cards = soup.find_all('div', class_='cNewsBox')
                print(
                    f"\t--> Found {len(cards)} articles on page {page} of '{category}'.")
                success_count = 0
                for card in cards:
                    try:
                        div = card.find('div', class_='row')
                        div = div.find('div', class_='col-md-8')
                        div = div.find('h3')
                        headline = div.find('a').get_text(
                            strip=True).replace('\xa0', ' ')
                        link = "https://urdu.dunyanews.tv/" + \
                            div.find('a')['href']
                        article_response = requests.get(link)
                        article_response.raise_for_status()
                        content_soup = BeautifulSoup(
                            article_response.text, "html.parser")
                        paras = content_soup.find(
                            'div', class_='main-news col-md-12').find_all('p')
                        combined_text = " ".join(
                            p.get_text(strip=True).replace(
                                '\xa0', ' ').replace('\u200b', '')
                            for p in paras if p.get_text(strip=True)
                        )
                        dunya_df['id'].append(self.id)
                        dunya_df['title'].append(headline)
                        dunya_df['link'].append(link)
                        dunya_df['gold_label'].append(category.replace('Entertainment', 'entertainment').replace(
                            'Technology', 'science-technology').replace('World', 'world').replace('Sports', 'sports').replace('Business', 'business'))
                        dunya_df['content'].append(combined_text)
                        self.id += 1
                        success_count += 1
                    except Exception as e:
                        print(
                            f"\t--> Failed to scrape an article on page {page} of '{category}': {e}")
                print(
                    f"\t--> Successfully scraped {success_count} articles from page {page} of '{category}'.")
            print('')
        return pd.DataFrame(dunya_df)

    def get_jang_articles(self, max_pages=1):
        jang_df = {
            "id": [],
            "title": [],
            "link": [],
            "content": [],
            "gold_label": [],
        }
        base_url = 'https://jang.com.pk/'
        categories = ['entertainment', 'business', 'sports', 'health-science',
                      'world']
        for category in categories:
            for page in range(1, max_pages + 1):
                print(f"Scraping page {page} of category '{category}'...")
                url = f"{base_url}/category/latest-news/{category}"
                response = requests.get(url)
                if response.url != url:
                    print(f"Redirected from {url} to {response.url}")
                response.raise_for_status()
                soup = BeautifulSoup(response.text, "html.parser")
                cards = soup.find(
                    'ul', class_='scrollPaginationNew__').find_all('li')
                print(
                    f"\t--> Found {len(cards)} articles on page {page} of '{category}'.")
                success_count = 0
                for card in cards:
                    try:
                        div = card.find('div', class_='main-heading')
                        headline = div.find('a').get_text(
                            strip=True).replace('\xa0', ' ')
                        link = div.find('a')['href']
                        article_response = requests.get(link)
                        article_response.raise_for_status()
                        content_soup = BeautifulSoup(
                            article_response.text, "html.parser")
                        paras = content_soup.find(
                            'div', class_='detail_view_content').find_all('p')
                        combined_text = " ".join(
                            p.get_text(strip=True).replace(
                                '\xa0', ' ').replace('\u200b', '')
                            for p in paras if p.get_text(strip=True)
                        )
                        jang_df['id'].append(self.id)
                        jang_df['title'].append(headline)
                        jang_df['link'].append(link)
                        jang_df['gold_label'].append(category.replace(
                            'health-science', 'science-technology'))
                        jang_df['content'].append(combined_text)
                        self.id += 1
                        success_count += 1
                    except Exception as e:
                        print(
                            f"\t--> Failed to scrape an article on page {page} of '{category}': {e}")
                print(
                    f"\t--> Successfully scraped {success_count} articles from page {page} of '{category}'.")
            print('')
        return pd.DataFrame(jang_df)

    def get_geo_articles(self):
        geo_df = {
            "id": [],
            "title": [],
            "link": [],
            "content": [],
            "gold_label": [],
        }
        base_url = 'https://urdu.geo.tv'
        categories = {
            'entertainment': 'entertainment',
            'business': 'business',
            'sports': 'sports',
            'science-technology': 'science-technology',
            'world': 'world'
        }
        for category_url, category_label in categories.items():
            print(f"Scraping category '{category_label}'...")
            url = f"{base_url}/category/{category_url}"
            print(f"From URL: {url}")
            try:
                response = requests.get(url)
                if response.url != url:
                    print(f"Redirected from {url} to {response.url}")
                response.raise_for_status()
                soup = BeautifulSoup(response.text, "html.parser")
                articles = soup.find_all('li', class_='border-box')
                print(f"\t--> Found {len(articles)
                                     } articles in '{category_label}'.")
                success_count = 0
                for article in articles:
                    try:
                        link_elem = article.find('a', class_='open-section')
                        if not link_elem:
                            continue
                        title = link_elem.get('title', '').strip()
                        link = link_elem.get('href', '')
                        if not title or not link:
                            continue
                        article_response = requests.get(link)
                        article_response.raise_for_status()
                        content_soup = BeautifulSoup(
                            article_response.text, "html.parser")
                        content_div = content_soup.find(
                            'div', class_='content-area')
                        if content_div:
                            # Get all p tags
                            paragraphs = content_div.find_all('p')
                            combined_text = " ".join(
                                p.get_text(strip=True).replace(
                                    '\xa0', ' ').replace('\u200b', '')
                                for p in paragraphs if p.get_text(strip=True)
                            )
                            if combined_text:
                                geo_df['id'].append(self.id)
                                geo_df['title'].append(title)
                                geo_df['link'].append(link)
                                geo_df['gold_label'].append(category_label)
                                geo_df['content'].append(combined_text)
                                self.id += 1
                                success_count += 1
                    except Exception as e:
                        print(
                            f"\t--> Failed to scrape an article in '{category_label}': {e}")
                print(
                    f"\t--> Successfully scraped {success_count} articles from '{category_label}'.")
                time.sleep(1)
            except Exception as e:
                print(f"Failed to scrape category '{category_label}': {e}")
            print('')
        return pd.DataFrame(geo_df)

    def get_samma_articles(self, max_pages=6):
        samma_df = {
            "id": [],
            "title": [],
            "link": [],
            "content": [],
            "gold_label": [],
        }
        base_url = 'https://urdu.samaa.tv'
        categories = ['lifestyle', 'money', 'sports', 'tech', 'global']
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36'
        }
        for category in categories:
            for page in range(1, max_pages + 1):
                print(f"Scraping page {page} of category '{category}'...")
                url = f"{base_url}/{category}?page={page}"
                try:
                    response = requests.get(url, headers=headers)
                    if response.url != url:
                        print(f"Redirected from {url} to {response.url}")
                    response.raise_for_status()
                    soup = BeautifulSoup(response.text, "html.parser")
                    articles = soup.find_all('div', class_='text')
                    print(
                        f"\t--> Found {len(articles)} articles on page {page} of '{category}'.")
                    success_count = 0
                    for article in articles:
                        try:
                            headline = article.find('h3').find('a').get_text(
                                strip=True).replace('\xa0', ' ')
                            link = article.find('h3').find('a')['href']
                            if not link.startswith('http'):
                                link = base_url + link
                            article_response = requests.get(
                                link, headers=headers)
                            article_response.raise_for_status()
                            content_soup = BeautifulSoup(
                                article_response.text, "html.parser")
                            paras = content_soup.find(
                                'div', class_='article-content').find_all('p')
                            combined_text = " ".join(
                                p.get_text(strip=True).replace(
                                    '\xa0', ' ').replace('\u200b', '')
                                for p in paras if p.get_text(strip=True)
                            )
                            samma_df['id'].append(self.id)
                            samma_df['title'].append(headline)
                            samma_df['link'].append(link)
                            samma_df['gold_label'].append(category.replace('lifestyle', 'entertainment').replace(
                                'tech', 'science-technology').replace('global', 'world').replace('sports', 'sports').replace('money', 'business'))
                            samma_df['content'].append(combined_text)
                            self.id += 1
                            success_count += 1
                        except Exception as e:
                            print(
                                f"\t--> Failed to scrape an article on page {page} of '{category}': {e}")
                    print(
                        f"\t--> Successfully scraped {success_count} articles from page {page} of '{category}'.")
                    time.sleep(3)
                except Exception as e:
                    print(f"Failed to scrape page {
                          page} of category '{category}': {e}")
            print('')
        return pd.DataFrame(samma_df)

In [15]:
scraper = NewsScraper()

In [16]:
express_df = scraper.get_express_articles()

Scraping page 1 of category 'saqafat'...
	--> Found 10 articles on page 1 of 'saqafat'.
	--> Successfully scraped 10 articles from page 1 of 'saqafat'.
Scraping page 2 of category 'saqafat'...
	--> Found 10 articles on page 2 of 'saqafat'.
	--> Successfully scraped 10 articles from page 2 of 'saqafat'.
Scraping page 3 of category 'saqafat'...
	--> Found 10 articles on page 3 of 'saqafat'.
	--> Successfully scraped 10 articles from page 3 of 'saqafat'.
Scraping page 4 of category 'saqafat'...
	--> Found 10 articles on page 4 of 'saqafat'.
	--> Successfully scraped 10 articles from page 4 of 'saqafat'.
Scraping page 5 of category 'saqafat'...
	--> Found 10 articles on page 5 of 'saqafat'.
	--> Successfully scraped 10 articles from page 5 of 'saqafat'.
Scraping page 6 of category 'saqafat'...
	--> Found 10 articles on page 6 of 'saqafat'.
	--> Successfully scraped 10 articles from page 6 of 'saqafat'.
Scraping page 7 of category 'saqafat'...
	--> Found 10 articles on page 7 of 'saqafat'.


In [17]:
dunya_df = scraper.get_dunya_articles()

Scraping page 1 of category 'Entertainment'...
	--> Found 18 articles on page 1 of 'Entertainment'.
	--> Successfully scraped 18 articles from page 1 of 'Entertainment'.

Scraping page 1 of category 'Business'...
	--> Found 18 articles on page 1 of 'Business'.
	--> Successfully scraped 18 articles from page 1 of 'Business'.

Scraping page 1 of category 'Sports'...
	--> Found 18 articles on page 1 of 'Sports'.
	--> Successfully scraped 18 articles from page 1 of 'Sports'.

Scraping page 1 of category 'Technology'...
	--> Found 18 articles on page 1 of 'Technology'.
	--> Successfully scraped 18 articles from page 1 of 'Technology'.

Scraping page 1 of category 'World'...
	--> Found 18 articles on page 1 of 'World'.
	--> Successfully scraped 18 articles from page 1 of 'World'.



In [18]:
jang_df = scraper.get_jang_articles()

Scraping page 1 of category 'entertainment'...
	--> Found 101 articles on page 1 of 'entertainment'.
	--> Failed to scrape an article on page 1 of 'entertainment': 'NoneType' object has no attribute 'find'
	--> Failed to scrape an article on page 1 of 'entertainment': 'NoneType' object has no attribute 'find'
	--> Successfully scraped 99 articles from page 1 of 'entertainment'.

Scraping page 1 of category 'business'...
	--> Found 99 articles on page 1 of 'business'.
	--> Failed to scrape an article on page 1 of 'business': 'NoneType' object has no attribute 'find'
	--> Failed to scrape an article on page 1 of 'business': 'NoneType' object has no attribute 'find'
	--> Failed to scrape an article on page 1 of 'business': HTTPSConnectionPool(host='jang.com.pk', port=443): Max retries exceeded with url: /news/1406756 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x0000025337EA38F0>: Failed to resolve 'jang.com.pk' ([Errno 11001] getaddrinfo failed)"))
	--> 

In [19]:
geo_df = scraper.get_geo_articles()

Scraping category 'entertainment'...
From URL: https://urdu.geo.tv/category/entertainment
	--> Found 60 articles in 'entertainment'.
	--> Successfully scraped 60 articles from 'entertainment'.

Scraping category 'business'...
From URL: https://urdu.geo.tv/category/business
	--> Found 60 articles in 'business'.
	--> Successfully scraped 60 articles from 'business'.

Scraping category 'sports'...
From URL: https://urdu.geo.tv/category/sports
	--> Found 60 articles in 'sports'.
	--> Successfully scraped 60 articles from 'sports'.

Scraping category 'science-technology'...
From URL: https://urdu.geo.tv/category/science-technology
	--> Found 60 articles in 'science-technology'.
	--> Successfully scraped 60 articles from 'science-technology'.

Scraping category 'world'...
From URL: https://urdu.geo.tv/category/world
	--> Found 60 articles in 'world'.
	--> Successfully scraped 60 articles from 'world'.



In [20]:
samaa_df = scraper.get_samma_articles()

Scraping page 1 of category 'lifestyle'...
	--> Found 25 articles on page 1 of 'lifestyle'.
	--> Successfully scraped 25 articles from page 1 of 'lifestyle'.
Scraping page 2 of category 'lifestyle'...
	--> Found 25 articles on page 2 of 'lifestyle'.
	--> Successfully scraped 25 articles from page 2 of 'lifestyle'.
Scraping page 3 of category 'lifestyle'...
	--> Found 25 articles on page 3 of 'lifestyle'.
	--> Successfully scraped 25 articles from page 3 of 'lifestyle'.
Scraping page 4 of category 'lifestyle'...
	--> Found 25 articles on page 4 of 'lifestyle'.
	--> Successfully scraped 25 articles from page 4 of 'lifestyle'.
Scraping page 5 of category 'lifestyle'...
	--> Found 25 articles on page 5 of 'lifestyle'.
	--> Successfully scraped 25 articles from page 5 of 'lifestyle'.
Scraping page 6 of category 'lifestyle'...
	--> Found 25 articles on page 6 of 'lifestyle'.
	--> Successfully scraped 25 articles from page 6 of 'lifestyle'.

Scraping page 1 of category 'money'...
	--> Found 2

In [21]:
geo_df.size/5

300.0

In [22]:
jang_df.size/5

492.0

In [23]:
dunya_df.size/5

90.0

In [24]:
express_df.size/5

700.0

In [25]:
samaa_df.size/5

750.0

In [28]:
print("The total size of the dataset is:", 2332)

The total size of the dataset is: 2332


# Output

- Save a combined csv of all 3 sites.


In [27]:
import pandas as pd
e_df = pd.DataFrame(express_df)
d_df = pd.DataFrame(dunya_df)
j_df = pd.DataFrame(jang_df)
g_df = pd.DataFrame(geo_df)
s_df = pd.DataFrame(samaa_df)
df_list = [e_df, d_df, j_df, g_df, s_df]
combine_df = pd.concat(df_list, ignore_index=True)
combine_df.to_csv('Group20_RawData.csv', index=False)